# When Models Lie to Please: Google Colab Runner

This notebook runs the actual, GPU-heavy 4-phase sparse interpretability research pipeline on a high-VRAM Google Colab runtime (T4, L4, or A100 GPU).

### Pipeline Steps:
1. **Git Clone/Pull**: Sets up the codebase and installs all dependencies.
2. **Build Datasets**: Generates the paired logic/opinion/sycophancy datasets.
3. **Phase 1 (Feature Discovery)**: Extracts activations and finds differential SAE features.
4. **Phase 2 (Circuit Tracing)**: Builds transcoder attribution graphs and analyzes divergence.
5. **Phase 3 (Transfer Testing)**: Runs cross-condition transfer and measures representation geometry.
6. **Phase 4 (Interventions & CAST)**: Trains classifiers and tests steering/CAST mitigations.
7. **Zip & Download**: Compresses results and datasets into a ZIP file for local download.

## Step 1: Install Dependencies and Setup

In [ ]:
# Clone repo or pull latest updates
!git clone https://github.com/ashioyajotham/when_models_lie_to_please.git || (cd when_models_lie_to_please && git pull)
%cd when_models_lie_to_please

# Install dependencies
!pip install -e ".[dev]"
!pip install git+https://github.com/google-deepmind/mishax.git

print("Setup complete!")

## Step 2: Configure Token and Run Pipeline
Make sure to add your HuggingFace token as a secret in Colab (named `HF_TOKEN`) or enter it when prompted.

In [ ]:
import os
import glob

# Authenticate with HuggingFace (required for gated Gemma 3 models)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = input("Enter your HuggingFace Token: ").strip()

# Step 0: Build paired prompt datasets
!python experiments/scripts/build_datasets.py --output-dir data/processed --min-pairs 500

# Clear any old/corrupted activation cache to ensure fresh extraction with loaded SAEs
!rm -rf data/activations/gemma3_4b

# Step 1: Feature discovery
!python experiments/scripts/run_phase1.py --config configs/experiment_configs/phase1_features.yaml --ignore-cache

# Find latest Phase 1 run_id to pass into Phase 2/3/4
phase1_runs = sorted(glob.glob("experiments/results/phase1/*"))
if not phase1_runs:
    raise RuntimeError("Phase 1 execution did not produce any output folders.")
run_id = os.path.basename(phase1_runs[-1])
print(f"\n>>> Detected Phase 1 Run ID: {run_id} <<<\n")

# Step 2: Circuit tracing
!python experiments/scripts/run_phase2.py --config configs/experiment_configs/phase2_circuits.yaml --phase1-run {run_id}

# Step 3: Shared mechanism testing
!python experiments/scripts/run_phase3.py --config configs/experiment_configs/phase3_transfer.yaml --phase1-run {run_id}

# Step 4: Detection and mitigation
!python experiments/scripts/run_phase4.py --config configs/experiment_configs/phase4_interventions.yaml --phase1-run {run_id} --phase3-run {run_id}

print("\nPipeline execution finished successfully!")

## Step 3: Zip and Download Results

In [ ]:
import zipfile
import datetime
from google.colab import files

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"colab_results_{timestamp}.zip"

print(f"Packaging results into {zip_filename}...")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Compress experiments results
    for root, dirs, files_in_dir in os.walk("experiments/results"):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file))
    
    # Compress processed datasets
    for root, dirs, files_in_dir in os.walk("data/processed"):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file))

print("Zipping finished. Triggering download in browser...")
files.download(zip_filename)